In [1]:

import json
import pennylane as qml
import pennylane.numpy as np

def U():
    """U function that creates the gate that checks the parity of the number of forests.
    It should not return anything, you simply need to add the gates."""

    # The input configuration is on qubits 0 to 7.
    # Qubit 8 is the auxiliary qubit used to store the parity, initialized to |0> by default.

    # Rule 1: If the first qubit (q_0) is |1>, it marks the start of a forest.
    # Flip the parity qubit (q_8) if q_0 is |1>.
    qml.CNOT(wires=[0, 8])

    # Rule 2: For subsequent qubits (q_1 to q_7), a forest starts if
    # the current qubit (q_i) is |1> AND the previous qubit (q_{i-1}) is |0>.
    # This corresponds to the pattern |01>.
    for i in range(1, 8):
        # To detect the |0> state on q_{i-1}, we apply an X gate to it.
        # This flips |0> to |1> and |1> to |0>.
        qml.PauliX(wires=i-1)

        # Now, if the original q_{i-1} was |0|, then q_{i-1} is currently |1>.
        # We use a Toffoli (CCNOT) gate with q_{i-1} and q_i as controls, and q_8 as target.
        # This gate flips q_8 if both control qubits are |1>.
        # So, q_8 is flipped if (original q_{i-1} was |0|) AND (q_i is |1>).
        qml.Toffoli(wires=[i-1, i, 8])

        # To ensure the input qubits are returned to their original state (uncompute),
        # apply another X gate to q_{i-1}. This reverts it to its initial value.
        qml.PauliX(wires=i-1)


# These functions are responsible for testing the solution.
def run(test_case_input: str) -> str:

    input = json.loads(test_case_input)
    wires_input = [0,1,2,3,4,5,6,7]

    # The device is set up with 10 wires (0-7 for input, 8 for parity, 9 unused).
    dev = qml.device("default.qubit", wires = 10, shots = 10)

    @qml.qnode(dev)
    def circuit():
      # Embed the input configuration into the first 8 qubits.
      qml.BasisEmbedding(input, wires = wires_input)

      # Apply the U operator to determine the parity of forests.
      U()

      # Measure the probability of the parity qubit (wire 8) being in the |1> state.
      # If the parity is odd, the probability of |1> will be 1.0.
      # If the parity is even, the probability of |1> will be 0.0.
      return qml.probs(wires = 8)

    # circuit()[1] gives the probability of wire 8 being |1>.
    return str(float(circuit()[1]))


def check(have: str, want: str) -> None:

    assert np.isclose(float(have), float(want)), "Wrong answer!"


# These are the public test cases
test_cases = [
    ('[1,0,1,1,0,1,1,1]', '1'),
    ('[0,0,0,0,0,1,0,1]', '0')
]

# This will run the public test cases locally
for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        if message := check(output, expected_output):
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")

        else:
            print("Correct!")



Running test case 0 with input '[1,0,1,1,0,1,1,1]'...
Correct!
Running test case 1 with input '[0,0,0,0,0,1,0,1]'...
Correct!
